In [1]:
import numpy as np
from ase import Atoms
from collections import Counter

TYPE_TO_ELEMENT = {13: "Al", 27: "Co", 28: "Ni"}

def parse(path):
    lines = open(path).read().splitlines()
    cell_rows, body_start = [], None
    for i, ln in enumerate(lines):
        if ln.strip() == "":
            continue
        cell_rows.append([float(x) for x in ln.split()[:3]])
        if len(cell_rows) == 3:
            body_start = i + 1
            break
    cell = np.array(cell_rows)
    symbols, scaled, dropped = [], [], 0
    for ln in lines[body_start:]:
        s = ln.strip()
        if s == "" or s.startswith("#"):
            continue
        parts = s.split()
        if len(parts) >= 2 and parts[1] == "#":
            continue
        if len(parts) < 5:
            continue
        try:
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
            typ = int(parts[3])
        except ValueError:
            continue
        if typ == 0:
            dropped += 1
            continue
        if typ in TYPE_TO_ELEMENT:
            symbols.append(TYPE_TO_ELEMENT[typ]); scaled.append([x, y, z])
    return Atoms(symbols=symbols, scaled_positions=np.array(scaled), cell=cell, pbc=True), dropped

def parse_fig5(path):
    lines = open(path).read().splitlines()
    text = open(path).read().split()
    cell = np.array([float(x) for x in text[:9]]).reshape(3, 3)
    symbols, scaled = [], []
    for ln in lines:
        parts = ln.split()
        if len(parts) >= 6 and parts[3].isdigit() and parts[4].isdigit():
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
            typ = int(parts[3])
            if typ == 0:
                continue
            if typ in TYPE_TO_ELEMENT:
                symbols.append(TYPE_TO_ELEMENT[typ]); scaled.append([x, y, z])
    return Atoms(symbols=symbols, scaled_positions=np.array(scaled), cell=cell, pbc=True)

print("Parsers ready.")

Parsers ready.


In [2]:
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter

def relax_with_diagnostics(structure_name, n_atoms, model_name, calc, base_atoms):
    a = base_atoms.copy()
    a.calc = calc
    e_initial = a.get_potential_energy()
    f_initial = a.get_forces()
    max_force_initial = np.sqrt((f_initial**2).sum(axis=1)).max()
    v0 = a.get_volume()
    p0 = a.get_positions().copy()
    opt = BFGS(FrechetCellFilter(a), logfile=None)
    opt.run(fmax=0.01)
    e_final = a.get_potential_energy()
    n = len(a)
    disp = np.sqrt(((a.get_positions() - p0)**2).sum(axis=1))
    return {
        "structure": structure_name, "n_atoms": n_atoms, "model": model_name,
        "dE_per_atom_meV": round(1000*(e_final - e_initial)/n, 1),
        "max_force_initial": round(max_force_initial, 3),
        "vol_change_pct": round(100*(a.get_volume()-v0)/v0, 2),
        "max_disp_A": round(disp.max(), 3),
    }

print("Diagnostics function ready.")

Diagnostics function ready.


In [3]:
from mattersim.forcefield import MatterSimCalculator
calc_ms = MatterSimCalculator(device="cpu")

def get_atoms(parse_result):
    if isinstance(parse_result, tuple):
        return parse_result[0]
    return parse_result

mattersim_results = []

a26 = get_atoms(parse_fig5("Al9Co2Ni2-coords.txt"))
mattersim_results.append(relax_with_diagnostics("Al9Co2Ni2", 26, "MatterSim-v1", calc_ms, a26))
print("26-atom done")

a60 = get_atoms(parse("Al2CoNi-coords.txt"))
mattersim_results.append(relax_with_diagnostics("Al2CoNi", 60, "MatterSim-v1", calc_ms, a60))
print("60-atom done")

a265 = get_atoms(parse("W-AlCoNi-coords"))
mattersim_results.append(relax_with_diagnostics("W-AlCoNi", 265, "MatterSim-v1", calc_ms, a265))
print("265-atom done")

import json
clean = [{k: (float(v) if hasattr(v, "item") else v) for k, v in r.items()} for r in mattersim_results]
with open("o1_mattersim_results.json", "w") as f:
    json.dump(clean, f, indent=2)

import pandas as pd
pd.DataFrame(clean)

/opt/anaconda3/envs/mattersim-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-28 14:33:58.126 | INFO     | mattersim.forcefield.potential:from_checkpoint:873 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
26-atom done


/opt/anaconda3/envs/mattersim-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.226722841044345e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/mattersim-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.274310930933957e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/mattersim-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.234799373870273e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/mattersim-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.281124272692217e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/mattersim-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be 

60-atom done
265-atom done


,structure,n_atoms,model,dE_per_atom_meV,max_force_initial,vol_change_pct,max_disp_A
0,Al9Co2Ni2,26,MatterSim-v1,-2.3,0.276,0.67,0.095
1,Al2CoNi,60,MatterSim-v1,-3.9,0.738,-0.14,0.083
2,W-AlCoNi,265,MatterSim-v1,-3.8,0.459,0.47,1.050


In [6]:
import json
import pandas as pd

mace = json.load(open("o1_table_v2.json"))
sevennet = json.load(open("o1_sevennet_results.json"))
mattersim = json.load(open("o1_mattersim_results.json"))

all_rows = mace + sevennet + mattersim
df = pd.DataFrame(all_rows)

# kappa_SRME from the live Matbench Discovery leaderboard (verified)
# Note: plain 7net-0 is not separately listed on the current board;
# its successor SevenNet-l3i5 sits at 0.550.
ksrme = {"MACE-MPA-0": 0.412, "MatterSim-v1": 0.575, "SevenNet-0": 0.550}
df["kappa_SRME"] = df["model"].map(ksrme)

df = df.sort_values(["n_atoms", "model"]).reset_index(drop=True)
df

,structure,n_atoms,model,dE_per_atom_meV,max_force_initial,vol_change_pct,max_disp_A,kappa_SRME
0,Al9Co2Ni2,26,MACE-MPA-0,-1.7,0.177,-1.13,0.060,0.412
1,Al9Co2Ni2,26,MatterSim-v1,-2.3,0.276,0.67,0.095,0.575
2,Al9Co2Ni2,26,SevenNet-0,-2.0,0.198,0.21,0.161,0.550
3,Al2CoNi,60,MACE-MPA-0,-1.3,0.330,0.31,0.045,0.412
4,Al2CoNi,60,MatterSim-v1,-3.9,0.738,-0.14,0.083,0.575
5,Al2CoNi,60,SevenNet-0,-1.9,0.335,0.38,0.060,0.550
6,W-AlCoNi,265,MACE-MPA-0,-1.9,0.297,-1.03,0.471,0.412
7,W-AlCoNi,265,MatterSim-v1,-3.8,0.459,0.47,1.050,0.575
8,W-AlCoNi,265,SevenNet-0,-2.4,0.316,0.40,0.583,0.550


The three models were chosen to span both architecture and the phonon-relevant
metric on the Matbench Discovery leaderboard, kappa_SRME (lower is better). MACE-MPA-0
sits at 0.412 and MatterSim-v1 at 0.575 on the current leaderboard. The plain
SevenNet-0 checkpoint I ran is an earlier model that is no longer listed separately
on the current board; its successor SevenNet-l3i5 sits at 0.550, which I use as the
indicative value for the SevenNet architecture. The point of the column is to show
the models span a range of this metric rather than all clustering at one value, not
to rank them precisely.

In [7]:
df.to_json("o1_final_table.json", orient="records", indent=2)
print("Saved final table:", len(df), "rows")

Saved final table: 9 rows
